In [302]:
#Imports 
import rasterio
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from lightgbm import LGBMClassifier,early_stopping, log_evaluation
import numpy as np

## Importing data:

In [303]:
back = rasterio.open("/kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/backscatter.tif")
bath = rasterio.open("/kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/bathymetry.tif")
train_df = pd.read_csv('/kaggle/input/competitions/geohab-mlwg-competition-2026/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/geohab-mlwg-competition-2026/test.csv')

In [304]:
print('Train set: ', train_df.shape)
display(train_df.head())

print('Test set: ', test_df.shape)
display(test_df.head())

back_data = back.read(1)
print('Backscatter data: ',back_data.shape)
print(back_data[:5, :5])

bath_data = bath.read(1)
print('Bathymestry data: ', bath_data.shape)
print(bath_data[:5, :5])

Train set:  (6256, 3)


,class,x,y
0,NVB,453594.477237,5.679192e+06
1,FMAT,453561.906453,5.679109e+06
2,ALG,453744.452238,5.679033e+06
3,ALG,453863.445302,5.679038e+06
4,ALG,453964.611906,5.679017e+06


Test set:  (98, 3)


,ID,x,y
0,1,453702.166779,5.679044e+06
1,2,454126.252800,5.678999e+06
2,3,453957.881092,5.678942e+06
3,4,453798.917484,5.678955e+06
4,5,453520.953671,5.679124e+06


Backscatter data:  (4040, 4743)
[[-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]]
Bathymestry data:  (4040, 4743)
[[-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]]


## Extracting Features:

In [305]:
def get_features(x, y):
    row, col = bath.index(x, y)
    return bath_data[row, col], back_data[row, col]

train_df[['depth', 'scatter']] = train_df.apply(
    lambda r: pd.Series(get_features(r['x'], r['y'])),
    axis=1
)

test_df[['depth', 'scatter']] = test_df.apply(
    lambda r: pd.Series(get_features(r['x'], r['y'])),
    axis=1
)

### Grid feature

In [306]:
def get_neighborhood_features(x, y):
    row, col = bath.index(x, y)
    
    features = []
    
    for dr in [-12, 0, 12]:
        for dc in [-12, 0, 12]:
            r = row + dr
            c = col + dc
            
            try:
                depth = bath_data[r, c]
                scatter = back_data[r, c]
            except:
                depth = 0
                scatter = 0
            
            features.append(depth)
            features.append(scatter)
    
    return features

In [307]:
features = train_df.apply(
    lambda r: get_neighborhood_features(r['x'], r['y']),
    axis=1
)

train_feature_cols = [f'f{i}' for i in range(len(features.iloc[0]))]

train_df[train_feature_cols] = pd.DataFrame(features.tolist(), index=train_df.index)

In [308]:
print(train_feature_cols)

['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f15', 'f16', 'f17']


In [309]:
train_df.head()

,class,x,y,depth,scatter,f0,f1,f2,f3,f4,...,f8,f9,f10,f11,f12,f13,f14,f15,f16,f17
0,NVB,453594.477237,5.679192e+06,-2.822883,-18.669088,-10000.000000,-10000.000000,-10000.000000,-10000.000000,-10000.000000,...,-2.822883,-18.669088,-3.003832,-16.779381,-3.052902,-17.719353,-3.153770,-18.979156,-3.217835,-21.499985
1,FMAT,453561.906453,5.679109e+06,-5.668992,-20.558794,-5.861867,-24.649494,-5.705794,-19.609058,-5.636959,...,-5.668992,-20.558794,-5.697957,-22.129887,-6.120852,-20.558794,-5.797802,-21.189917,-6.232965,-21.189917
2,ALG,453744.452238,5.679033e+06,-9.288988,-23.389690,-9.233783,-24.339426,-8.520893,-24.019592,-8.426841,...,-9.288988,-23.389690,-9.054879,-23.709524,-9.672694,-27.169104,-10.006989,-21.189917,-10.029820,-23.709524
3,ALG,453863.445302,5.679038e+06,-7.263793,-21.189917,-6.856915,-24.339426,-6.731852,-24.339426,-6.698797,...,-7.263793,-21.189917,-6.937677,-22.449720,-8.047904,-19.609058,-7.883994,-18.039186,-7.642729,-21.819818
4,ALG,453964.611906,5.679017e+06,-9.784807,-22.449720,-8.769996,-24.339426,-7.727922,-20.238960,-7.976172,...,-9.784807,-22.449720,-8.631302,-28.749962,-10.117740,-25.599232,-10.628894,-27.169104,-10.871862,-27.488937


### Advanced feature engineering:

In [310]:
def get_advanced_features(x, y):
    row, col = bath.index(x, y)
    
    depths = []
    scatters = []
    
    for dr in [-10, 0, 10]:
        for dc in [-10, 0, 10]:
            r = row + dr
            c = col + dc
            
            if 0 <= r < bath_data.shape[0] and 0 <= c < bath_data.shape[1]:
                depths.append(bath_data[r, c])
                scatters.append(back_data[r, c])
            else:
                depths.append(0)
                scatters.append(0)
    
    return [
        np.mean(depths),
        np.std(depths),
        np.min(depths),
        np.max(depths),
        np.mean(scatters),
        np.std(scatters),
        np.min(scatters),
        np.max(scatters),
    ]

In [311]:
features = train_df.apply(
    lambda r: get_advanced_features(r['x'], r['y']),
    axis=1
)

adv_feature_cols = [
    'depth_mean', 'depth_std', 'depth_min', 'depth_max',
    'scatter_mean', 'scatter_std', 'scatter_min', 'scatter_max'
]

train_df[adv_feature_cols] = pd.DataFrame(features.tolist(), index=train_df.index)

In [312]:
print(adv_feature_cols)

['depth_mean', 'depth_std', 'depth_min', 'depth_max', 'scatter_mean', 'scatter_std', 'scatter_min', 'scatter_max']


### Cluster Feature

In [313]:
# from sklearn.cluster import KMeans

# # fit
# kmeans = KMeans(n_clusters=7, random_state=42)
# kmeans.fit(train_df[['x', 'y', 'depth', 'scatter']])

# # transform
# train_dist = kmeans.transform(train_df[['x','y','depth','scatter']])
# test_dist = kmeans.transform(test_df[['x','y','depth','scatter']])

# cluster_feat = []

# # add features
# for i in range(train_dist.shape[1]):
#     train_df[f'cluster_dist_{i}'] = train_dist[:, i]
#     test_df[f'cluster_dist_{i}'] = test_dist[:, i]
#     cluster_feat.append(f'cluster_dist_{i}')

In [314]:
target = 'class'

X = train_df[train_feature_cols + adv_feature_cols + ['x','y','depth','scatter'] ]
y = train_df[target].map({'SGAM': 0, 'NVB': 1, 'SGZ': 2, 'ALG': 3, 'FMAT': 4})

## OOF preds

In [315]:
features = test_df.apply(
    lambda r: get_neighborhood_features(r['x'], r['y']),
    axis=1
)

test_feature_cols = [f'f{i}' for i in range(len(features.iloc[0]))]

test_df[test_feature_cols] = pd.DataFrame(features.tolist(), index=test_df.index)

In [316]:
features = test_df.apply(
    lambda r: get_advanced_features(r['x'], r['y']),
    axis=1
)

test_df[adv_feature_cols] = pd.DataFrame(features.tolist(), index=test_df.index)

In [317]:
X_test = test_df[test_feature_cols + adv_feature_cols + ['x','y','depth','scatter']]

In [318]:
# Hyperparameters found via Optuna
best_params = {
    "n_estimators": 2000,
    "learning_rate": 0.07918500203202344,
    "max_depth": 11,
    "num_leaves": 77,
    "subsample": 0.8055152296793877,
    "colsample_bytree": 0.7103689169741864,
    "reg_alpha": 0.014069050212386091,
    "reg_lambda": 0.7078441464734376,
    "min_child_samples": 9,
    "class_weight": "balanced",
    "random_state": 42,
    "device": "gpu",
    "n_jobs": -1,
    "verbose": -1,
}

oof_preds = np.empty(len(X), dtype=object)
test_probs = np.zeros((len(X_test), len(np.unique(y))))
fold_scores = []
classes = None
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

tr_feat = train_feature_cols + adv_feature_cols   
test_feat = test_feature_cols + adv_feature_cols

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx].copy(), y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx].copy(), y.iloc[val_idx]

    # KMeans fit on train fold only
    kmeans = KMeans(n_clusters=7, random_state=42)
    kmeans.fit(X_tr[['x', 'y', 'depth', 'scatter']])

    train_dist = kmeans.transform(X_tr[['x', 'y', 'depth', 'scatter']])
    val_dist   = kmeans.transform(X_va[['x', 'y', 'depth', 'scatter']])

    clust_feat = []
    for i in range(train_dist.shape[1]):
        X_tr[f'cluster_dist_{i}'] = train_dist[:, i]
        X_va[f'cluster_dist_{i}'] = val_dist[:, i]
        clust_feat.append(f'cluster_dist_{i}')

    all_feat = tr_feat + clust_feat

    model = LGBMClassifier(**best_params)
    model.fit(
        X_tr[all_feat], y_tr,
        eval_set=[(X_va[all_feat], y_va)],
        callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(0)]
    )

    val_preds = model.predict(X_va[all_feat])
    oof_preds[val_idx] = val_preds
    fold_f1 = f1_score(y_va, val_preds, average="weighted")
    fold_scores.append(fold_f1)
    print(f"Fold {fold+1} | Iterations: {model.best_iteration_:>3} | F1: {fold_f1:.4f}")

    test_dist = kmeans.transform(X_test[['x', 'y', 'depth', 'scatter']])
    X_test_fold = X_test[test_feat].copy()
    for i in range(test_dist.shape[1]):
        X_test_fold[f'cluster_dist_{i}'] = test_dist[:, i]

    if classes is None:
        classes = model.classes_
    test_probs += model.predict_proba(X_test_fold) / skf.n_splits

print("\n--- Overall OOF Evaluation ---")
print(f"Mean Fold F1: {np.mean(fold_scores):.4f} (±{np.std(fold_scores):.4f})")

Fold 1 | Iterations:  78 | F1: 0.9856
Fold 2 | Iterations: 164 | F1: 0.9904
Fold 3 | Iterations:  84 | F1: 0.9872
Fold 4 | Iterations: 148 | F1: 0.9824
Fold 5 | Iterations: 139 | F1: 0.9904

--- Overall OOF Evaluation ---
Mean Fold F1: 0.9872 (±0.0031)


In [319]:
final_preds = classes[np.argmax(test_probs, axis=1)]

submission = pd.DataFrame({
    'ID': test_df['ID'],
    target: final_preds
})

submission[target] = submission[target].map({0: 'SGAM', 1: 'NVB', 2: 'SGZ', 3: 'ALG', 4: 'FMAT'})

submission.to_csv('submission_clustering.csv', index=False)

Apparantly fitting the KMEans on training data and then predicting on training data to use it as a feature caused information leakage so this approach scores well on CV but the while making predictions on test set it fails harshly